In [1]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets hf_transfer

In [ ]:
# hf_transfer: tải model đa luồng, nhanh hơn hẳn client mặc định.
# Set biến môi trường "HF_TOKEN" khi tạo pod (RunPod: mục "Environment Variables" lúc deploy) —
# KHÔNG hardcode token vào file này, vì file nằm trong git repo.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import login

hf_token = ""
if hf_token:
    login(token=hf_token)
else:
    print("[WARN] Chưa set biến môi trường 'HF_TOKEN' → dùng unauthenticated request tới HF Hub (có thể bị rate limit).")


In [3]:
import json
import re
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback,
)
from trl import SFTConfig, SFTTrainer

## Helper functions — đọc/ghi dữ liệu

In [4]:
def _get_records(jsonl_path: Path) -> list[dict]:
    """
    - Summary: Đọc toàn bộ record hợp lệ từ file JSONL.
    - Args:
        - jsonl_path: Đường dẫn file JSONL.
    - Output:
        - list[dict]: Danh sách record hợp lệ.
    """
    records: list[dict] = []
    with jsonl_path.open(encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass
    return records


def _write_jsonl(records: list[dict], out_path: Path):
    """
    - Summary: Ghi list record ra file JSONL, ghi đè.
    - Args:
        - records: List record cần ghi.
        - out_path: Đường dẫn file output JSONL.
    - Output:
        - None. Ghi file tại out_path.
    """
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open('w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')


def _get_prompt_prefix(prompt_path: Path | None) -> str:
    """
    - Summary: Đọc nội dung prompt cần chèn trước content.
    - Args:
        - prompt_path: Đường dẫn file prompt, None nếu không dùng.
    - Output:
        - str: Nội dung prompt, "" nếu prompt_path là None.
    """
    if prompt_path is None:
        return ""
    return prompt_path.read_text(encoding='utf-8')


def _get_safe_name(name: str) -> str:
    """
    - Summary: Chuẩn hoá tên model/dataset để đặt tên file an toàn.
    - Args:
        - name: Tên gốc, có thể chứa "/" (VD: "Qwen/Qwen2.5-7B-Instruct").
    - Output:
        - str: Tên đã thay ký tự không hợp lệ bằng "_".
    """
    return re.sub(r"[^\w\-.]", "_", name)


def _get_dataset_name(dataset_path: str) -> str:
    """
    - Summary: Lấy tên dataset từ đường dẫn file, bỏ phần mở rộng.
    - Args:
        - dataset_path: Đường dẫn file JSONL.
    - Output:
        - str: Tên file không kèm phần mở rộng.
    """
    return Path(dataset_path).stem

## Helper functions — prompt & dataset cho SFT

In [5]:
def _build_messages(content: str, prompt_prefix: str, target_events: list[dict] | None) -> list[dict]:
    """
    - Summary: Build list message theo chat template Qwen.
    - Args:
        - content: Nội dung văn bản cần trích xuất sự kiện.
        - prompt_prefix: Prompt hệ thống chèn trước content, "" nếu không có.
        - target_events: List event mục tiêu (label), None nếu là suy luận (infer).
    - Output:
        - list[dict]: List message [system?, user, assistant?].
    """
    messages: list[dict] = []
    if prompt_prefix:
        messages.append({"role": "system", "content": prompt_prefix})
    messages.append({"role": "user", "content": content})
    if target_events is not None:
        messages.append({"role": "assistant", "content": json.dumps(target_events, ensure_ascii=False)})
    return messages


def _build_sft_dataset(records: list[dict], prompt_prefix: str, tokenizer) -> Dataset:
    """
    - Summary:
        1. Build message theo chat template cho từng record (_build_messages()).
        2. Render message thành text hoàn chỉnh bằng tokenizer.
    - Args:
        - records: List record {content, events} đọc từ JSONL.
        - prompt_prefix: Prompt hệ thống chèn trước content.
        - tokenizer: Tokenizer của model, dùng apply_chat_template.
    - Output:
        - Dataset: HuggingFace Dataset với field "text" sẵn sàng cho SFTTrainer.
    """
    texts = [
        tokenizer.apply_chat_template(
            _build_messages(record["content"], prompt_prefix, record.get("events", [])),
            tokenize=False,
        )
        for record in records
    ]
    return Dataset.from_dict({"text": texts})

## Model + LoRA (QLoRA 4-bit)

In [6]:
def _build_model_and_tokenizer(
    model_name:          str,
    lora_r:              int,
    lora_alpha:          int,
    lora_dropout:        float,
    lora_target_modules: list[str],
):
    """
    - Summary:
        1. Load tokenizer + model 4-bit (QLoRA) từ model_name.
        2. Chuẩn bị model cho k-bit training.
        3. Gắn LoRA adapter theo config.
    - Args:
        - model_name: Tên model gốc trên HF Hub (VD: "Qwen/Qwen2.5-7B-Instruct").
        - lora_r: Rank của LoRA.
        - lora_alpha: Alpha của LoRA.
        - lora_dropout: Dropout của LoRA.
        - lora_target_modules: List tên module cần gắn LoRA.
    - Output:
        - tuple: (model đã gắn LoRA, tokenizer).
    """
    bnb_config = BitsAndBytesConfig(
        load_in_4bit              = True,
        bnb_4bit_quant_type       = "nf4",
        bnb_4bit_compute_dtype    = torch.bfloat16,
        bnb_4bit_use_double_quant = True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token  # Qwen không có pad_token riêng

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config = bnb_config,
        device_map          = "auto",
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    lora_config = LoraConfig(
        r              = lora_r,
        lora_alpha     = lora_alpha,
        lora_dropout   = lora_dropout,
        target_modules = lora_target_modules,
        bias           = "none",
        task_type      = "CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.config.use_cache = False  # bắt buộc tắt khi dùng gradient checkpointing
    return model, tokenizer

## Inference cho eval trên test_datasets

In [7]:
def _get_model_prediction(model, tokenizer, content: str, prompt_prefix: str, max_new_tokens: int) -> str:
    """
    - Summary: Sinh prediction cho 1 sample bằng model tại checkpoint hiện tại.
    - Args:
        - model: Model (đã gắn LoRA) đang train.
        - tokenizer: Tokenizer tương ứng.
        - content: Nội dung văn bản cần trích xuất sự kiện.
        - prompt_prefix: Prompt hệ thống chèn trước content.
        - max_new_tokens: Số token tối đa cần sinh.
    - Output:
        - str: Chuỗi raw model sinh ra (chưa parse JSON), để tự tính độ đo sau.
    """
    messages    = _build_messages(content, prompt_prefix, target_events=None)
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs      = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            pad_token_id   = tokenizer.pad_token_id,
        )
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def _build_predictions_for_dataset(
    model,
    tokenizer,
    records:        list[dict],
    prompt_prefix:  str,
    max_new_tokens: int,
) -> list[dict]:
    """
    - Summary:
        1. Bật lại cache để generate nhanh hơn (đã tắt lúc train).
        2. Infer từng record trong test dataset (_get_model_prediction()).
        3. Tắt cache lại để tiếp tục train.
    - Args:
        - model: Model đang train, dùng để infer.
        - tokenizer: Tokenizer tương ứng.
        - records: List record test dataset {id, content, ...}.
        - prompt_prefix: Prompt hệ thống chèn trước content.
        - max_new_tokens: Số token tối đa cần sinh.
    - Output:
        - list[dict]: List {id, predict} cho từng record, để tự tính độ đo sau.
    """
    model.config.use_cache = True
    predictions: list[dict] = []
    for record in records:
        predict = _get_model_prediction(model, tokenizer, record["content"], prompt_prefix, max_new_tokens)
        predictions.append({"id": record.get("id"), "predict": predict})
    model.config.use_cache = False
    return predictions

## Checkpoint: lưu weight + eval sau mỗi N batch

In [8]:
def _process_checkpoint(
    model,
    tokenizer,
    step:               int,
    model_name:         str,
    train_dataset_name: str,
    test_datasets:      list[tuple[str, list[dict]]],
    prompt_prefix:      str,
    max_new_tokens:     int,
    output_dir:         Path,
):
    """
    - Summary:
        1. Lưu adapter weight tại checkpoint hiện tại.
        2. Infer trên từng test dataset (_build_predictions_for_dataset()).
        3. Ghi predict ra file JSONL riêng từng test dataset (_write_jsonl()).
    - Args:
        - model: Model (đã gắn LoRA) tại checkpoint hiện tại.
        - tokenizer: Tokenizer tương ứng.
        - step: Số batch (global_step) hiện tại.
        - model_name: Tên model gốc, dùng để đặt tên file.
        - train_dataset_name: Tên dataset train, dùng để đặt tên file weight.
        - test_datasets: List (tên dataset, records) cần eval.
        - prompt_prefix: Prompt hệ thống chèn trước content.
        - max_new_tokens: Số token tối đa cần sinh khi infer.
        - output_dir: Thư mục gốc chứa weight + predict.
    - Output:
        - None. Ghi weight + predict ra output_dir.
    """
    safe_model = _get_safe_name(model_name)

    weight_name = f"{safe_model}_{train_dataset_name}_{step}"
    model.save_pretrained(output_dir / weight_name)
    print(f"[CKPT {step}] đã lưu weight: {weight_name}")

    #for test_dataset_name, test_records in test_datasets:
    #    predictions  = _build_predictions_for_dataset(model, tokenizer, test_records, prompt_prefix, max_new_tokens)
    #    predict_name = f"{safe_model}_{test_dataset_name}_{step}_predict.jsonl"
    #    _write_jsonl(predictions, output_dir / predict_name)
    #    print(f"[CKPT {step}] đã infer '{test_dataset_name}': {len(predictions)} sample → {predict_name}")


class CheckpointEvalCallback(TrainerCallback):
    """
    - Summary: Sau mỗi save_every_n_steps batch, lưu weight + infer test_datasets (_process_checkpoint()).
    - Args:
        - model_name: Tên model gốc, dùng để đặt tên file.
        - train_dataset_name: Tên dataset train.
        - test_datasets: List (tên dataset, records) cần eval.
        - tokenizer: Tokenizer dùng khi infer.
        - prompt_prefix: Prompt hệ thống chèn trước content.
        - max_new_tokens: Số token tối đa cần sinh khi infer.
        - output_dir: Thư mục gốc chứa weight + predict.
        - save_every_n_steps: Số batch giữa 2 lần lưu weight + eval.
    """

    def __init__(
        self,
        model_name:         str,
        train_dataset_name: str,
        test_datasets:      list[tuple[str, list[dict]]],
        tokenizer,
        prompt_prefix:      str,
        max_new_tokens:     int,
        output_dir:         Path,
        save_every_n_steps: int,
    ):
        self.model_name         = model_name
        self.train_dataset_name = train_dataset_name
        self.test_datasets      = test_datasets
        self.tokenizer          = tokenizer
        self.prompt_prefix      = prompt_prefix
        self.max_new_tokens     = max_new_tokens
        self.output_dir         = output_dir
        self.save_every_n_steps = save_every_n_steps

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step == 0 or state.global_step % self.save_every_n_steps != 0:
            return control

        model.eval()
        _process_checkpoint(
            model               = model,
            tokenizer           = self.tokenizer,
            step                = state.global_step,
            model_name          = self.model_name,
            train_dataset_name  = self.train_dataset_name,
            test_datasets       = self.test_datasets,
            prompt_prefix       = self.prompt_prefix,
            max_new_tokens      = self.max_new_tokens,
            output_dir          = self.output_dir,
        )
        model.train()
        return control

## Log loss ra file (đọc được kể cả khi mất kết nối notebook, session vẫn chạy nền)

In [9]:
class FileLoggingCallback(TrainerCallback):
    """
    - Summary: Mỗi lần Trainer log (theo logging_steps), append log ra file JSONL.
    - Args:
        - log_path: Đường dẫn file JSONL ghi log (loss, learning_rate, epoch...).
    """

    def __init__(self, log_path: Path):
        self.log_path = log_path
        self.log_path.parent.mkdir(parents=True, exist_ok=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return control

        record = {"step": state.global_step, **logs}
        with self.log_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
        return control

## Orchestrator train

In [10]:
def run_training(
    model_name:                  str,
    train_dataset_path:          str,
    test_dataset_paths:          list[str],
    prompt_path:                 Path | None,
    output_dir:                  Path,
    lora_r:                      int,
    lora_alpha:                  int,
    lora_dropout:                float,
    lora_target_modules:         list[str],
    num_train_epochs:            int,
    per_device_batch_size:       int,
    gradient_accumulation_steps: int,
    learning_rate:               float,
    max_seq_length:              int,
    save_every_n_steps:          int,
    max_new_tokens:              int,
):
    """
    - Summary:
        1. Đọc prompt prefix (_get_prompt_prefix()).
        2. Build model + tokenizer gắn LoRA (_build_model_and_tokenizer()).
        3. Đọc train + test_datasets, build SFT dataset (_build_sft_dataset()).
        4. Train với SFTTrainer, checkpoint + eval qua CheckpointEvalCallback.
        5. Log loss ra file JSONL qua FileLoggingCallback.
    - Args:
        - model_name: Tên model gốc trên HF Hub.
        - train_dataset_path: Đường dẫn file JSONL train.
        - test_dataset_paths: List đường dẫn file JSONL cần eval.
        - prompt_path: Đường dẫn file prompt chèn trước content, None nếu không dùng.
        - output_dir: Thư mục ghi weight + predict.
        - lora_r, lora_alpha, lora_dropout, lora_target_modules: Config LoRA.
        - num_train_epochs, per_device_batch_size, gradient_accumulation_steps,
          learning_rate, max_seq_length: Config train.
        - save_every_n_steps: Số batch giữa 2 lần lưu weight + eval.
        - max_new_tokens: Số token tối đa cần sinh khi infer test_datasets.
    - Output:
        - None. Weight + predict được ghi liên tục vào output_dir trong lúc train.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    log_path = output_dir / "train_loss.jsonl"

    prompt_prefix    = _get_prompt_prefix(prompt_path)
    model, tokenizer = _build_model_and_tokenizer(model_name, lora_r, lora_alpha, lora_dropout, lora_target_modules)

    train_records      = _get_records(Path(train_dataset_path))
    train_dataset_name = _get_dataset_name(train_dataset_path)
    sft_dataset        = _build_sft_dataset(train_records, prompt_prefix, tokenizer)

    test_datasets = [
        (_get_dataset_name(test_path), _get_records(Path(test_path)))
        for test_path in test_dataset_paths
    ]

    training_args = SFTConfig(
        output_dir                  = str(output_dir / "trainer_state"),
        num_train_epochs            = num_train_epochs,
        per_device_train_batch_size = per_device_batch_size,
        gradient_accumulation_steps = gradient_accumulation_steps,
        learning_rate               = learning_rate,
        max_length                  = max_seq_length,
        dataset_text_field          = "text",
        packing                     = False,
        logging_steps               = 8,
        save_strategy               = "no",  # tự lưu qua CheckpointEvalCallback, không dùng cơ chế save mặc định
        bf16                        = torch.cuda.is_bf16_supported(),
        report_to                   = "none",
    )

    checkpoint_callback = CheckpointEvalCallback(
        model_name         = model_name,
        train_dataset_name = train_dataset_name,
        test_datasets       = test_datasets,
        tokenizer           = tokenizer,
        prompt_prefix       = prompt_prefix,
        max_new_tokens      = max_new_tokens,
        output_dir          = output_dir,
        save_every_n_steps  = save_every_n_steps,
    )
    file_logging_callback = FileLoggingCallback(log_path=log_path)

    print(f"[LOG] loss sẽ được ghi liên tục tại: {log_path}")

    trainer = SFTTrainer(
        model         = model,
        args          = training_args,
        train_dataset = sft_dataset,
        processing_class     = tokenizer,
        callbacks     = [checkpoint_callback, file_logging_callback],
    )
    trainer.train()

## Config & chạy

Sửa đường dẫn `train_dataset` / `test_datasets` / `prompt_path` theo Kaggle Dataset đã upload.

In [11]:
config = {
    "model_name": "Qwen/Qwen2.5-3B-Instruct",
    "train_dataset": "/workspace/augmented_train.jsonl",
    "test_datasets": [
        "/workspace/train_v1_subset.jsonl",
    ],
    "prompt_path": "/workspace/study_prompt.v2.txt",  # VD: "/kaggle/input/dataset/instruction_prompt.txt", None nếu không chèn prompt
    "output_dir": "/workspace/output",
    "lora": {
        "r":              16,
        "alpha":          32,
        "dropout":        0.05,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    },
    "train": {
        "num_train_epochs":            3,
        "per_device_batch_size":       8,
        "gradient_accumulation_steps": 8,
        "learning_rate":               2e-4,
        "max_seq_length":              2048,
        "save_every_n_steps":          16,  # số batch giữa 2 lần lưu weight + eval trên test_datasets
    },
    "max_new_tokens": 512,
}

model_name         = config["model_name"]
train_dataset_path = config["train_dataset"]
test_dataset_paths = config["test_datasets"]
prompt_path        = Path(config["prompt_path"]) if config["prompt_path"] else None
output_dir         = Path(config["output_dir"])
lora_cfg           = config["lora"]
train_cfg          = config["train"]

In [12]:
import torch
import gc

# Xóa các biến model cũ nếu đã lỡ khởi tạo
if 'model' in locals():
    del model
if 'trainer' in locals():
    del trainer

torch.cuda.empty_cache()
torch.cuda.ipc_collect()
gc.collect()

75

In [ ]:
run_training(
    model_name                  = model_name,
    train_dataset_path          = train_dataset_path,
    test_dataset_paths          = test_dataset_paths,
    prompt_path                 = prompt_path,
    output_dir                  = output_dir,
    lora_r                      = lora_cfg["r"],
    lora_alpha                  = lora_cfg["alpha"],
    lora_dropout                = lora_cfg["dropout"],
    lora_target_modules         = lora_cfg["target_modules"],
    num_train_epochs            = train_cfg["num_train_epochs"],
    per_device_batch_size       = train_cfg["per_device_batch_size"],
    gradient_accumulation_steps = train_cfg["gradient_accumulation_steps"],
    learning_rate               = train_cfg["learning_rate"],
    max_seq_length              = train_cfg["max_seq_length"],
    save_every_n_steps          = train_cfg["save_every_n_steps"],
    max_new_tokens              = config["max_new_tokens"],
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Building labels for train dataset:   0%|          | 0/10650 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10650 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
8,1.596500
16,1.122811
24,0.567402
32,0.143381
40,0.028387
48,0.010846
56,0.005558
64,0.002219
72,0.001499
80,0.001254


[CKPT 16] đã lưu weight: Qwen_Qwen2.5-3B-Instruct_augmented_train_16
[CKPT 32] đã lưu weight: Qwen_Qwen2.5-3B-Instruct_augmented_train_32
[CKPT 48] đã lưu weight: Qwen_Qwen2.5-3B-Instruct_augmented_train_48
[CKPT 64] đã lưu weight: Qwen_Qwen2.5-3B-Instruct_augmented_train_64
[CKPT 80] đã lưu weight: Qwen_Qwen2.5-3B-Instruct_augmented_train_80
[CKPT 96] đã lưu weight: Qwen_Qwen2.5-3B-Instruct_augmented_train_96
[CKPT 112] đã lưu weight: Qwen_Qwen2.5-3B-Instruct_augmented_train_112
[CKPT 128] đã lưu weight: Qwen_Qwen2.5-3B-Instruct_augmented_train_128
[CKPT 144] đã lưu weight: Qwen_Qwen2.5-3B-Instruct_augmented_train_144
[CKPT 160] đã lưu weight: Qwen_Qwen2.5-3B-Instruct_augmented_train_160
[CKPT 176] đã lưu weight: Qwen_Qwen2.5-3B-Instruct_augmented_train_176
[CKPT 192] đã lưu weight: Qwen_Qwen2.5-3B-Instruct_augmented_train_192
[CKPT 208] đã lưu weight: Qwen_Qwen2.5-3B-Instruct_augmented_train_208
[CKPT 224] đã lưu weight: Qwen_Qwen2.5-3B-Instruct_augmented_train_224
[CKPT 240] đã lưu 